In [ ]:
import torch
from pathlib import Path

In [ ]:
RAW_DATA_DIR: Path = Path('/shared/sets/datasets/vision/CUB-200')
PROC_DATA_DIR: Path = Path('/home/awrobel/quantization/data')
DATASET_DIR: Path = PROC_DATA_DIR/"dataset"
LOG_DIR: Path = Path("logs")

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
!nvidia-smi

# Crop and visualize dataset

In [ ]:
import os
from PIL import Image
from prepare_cub import crop_dataset

In [ ]:
crop_dataset(
    raw_data_dir=RAW_DATA_DIR,
    proc_data_dir=PROC_DATA_DIR,
)

In [ ]:
FOLD = "train"
CLASS = os.listdir(FOLD_PATH)[0]
CLASS_FOLD_PATH = DATASET_DIR/FOLD/CLASS

for img_fname in os.listdir(CLASS_FOLD_PATH):
    img = Image.open(CLASS_FOLD_PATH/img_fname)
    display(img)
    break

# Setup dataloaders

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from typing import Dict, Tuple

In [ ]:
BATCH_SIZE:  int = 32
NUM_WORKERS: int = 8

In [ ]:
class TrivialAugmentWideNoColor(transforms.TrivialAugmentWide):
    def _augmentation_space(self, num_bins: int) -> Dict[str, Tuple[torch.Tensor, bool]]:
        return {
            "Identity": (torch.tensor(0.0), False),
            "ShearX": (torch.linspace(0.0, 0.5, num_bins), True), 
            "ShearY": (torch.linspace(0.0, 0.5, num_bins), True), 
            "TranslateX": (torch.linspace(0.0, 16.0, num_bins), True), 
            "TranslateY": (torch.linspace(0.0, 16.0, num_bins), True), 
            "Rotate": (torch.linspace(0.0, 60.0, num_bins), True), 
        }


class TrivialAugmentWideNoShapeWithColor(transforms.TrivialAugmentWide):
    def _augmentation_space(self, num_bins: int) -> Dict[str, Tuple[torch.Tensor, bool]]:
        return {
            "Identity": (torch.tensor(0.0), False),
            "Brightness": (torch.linspace(0.0, 0.5, num_bins), True),
            "Color": (torch.linspace(0.0, 0.5, num_bins), True), 
            "Contrast": (torch.linspace(0.0, 0.5, num_bins), True),
            "Sharpness": (torch.linspace(0.0, 0.5, num_bins), True),
            "Posterize": (8 - (torch.arange(num_bins) / ((num_bins - 1) / 6)).round().int(), False),
            "Solarize": (torch.linspace(255.0, 0.0, num_bins), False),
            "AutoContrast": (torch.tensor(0.0), False),
            "Equalize": (torch.tensor(0.0), False),
        }


class TrivialAugmentWideNoShape(transforms.TrivialAugmentWide):
    def _augmentation_space(self, num_bins: int) -> Dict[str, Tuple[torch.Tensor, bool]]:
        return {
            
            "Identity": (torch.tensor(0.0), False),
            "Brightness": (torch.linspace(0.0, 0.5, num_bins), True),
            "Color": (torch.linspace(0.0, 0.02, num_bins), True), 
            "Contrast": (torch.linspace(0.0, 0.5, num_bins), True),
            "Sharpness": (torch.linspace(0.0, 0.5, num_bins), True),
            "Posterize": (8 - (torch.arange(num_bins) / ((num_bins - 1) / 6)).round().int(), False),
            "AutoContrast": (torch.tensor(0.0), False),
            "Equalize": (torch.tensor(0.0), False),
        }
# V1
# train_transforms = transforms.Compose([
#     transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(15),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
#     transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
#     transforms.RandomPerspective(distortion_scale=0.2, p=0.5, interpolation=3),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])

# V2
# train_transforms = transforms.Compose([
#     transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(15),
#     transforms.ColorJitter(brightness=0.2, contrast=0.5, saturation=0.5, hue=0.3),
#     transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
#     transforms.RandomPerspective(distortion_scale=0.2, p=0.5, interpolation=3),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#     transforms.RandomErasing(p=0.5, scale=(0.02, 0.25), ratio=(0.3, 3.3)),
# ])
# V3
# train_transforms = transforms.Compose([
#     transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(30),
#     transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.5),
#     transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.8, 1.2)),
#     transforms.RandomPerspective(distortion_scale=0.2, p=0.7, interpolation=3),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#     transforms.RandomErasing(p=0.6, scale=(0.02, 0.33), ratio=(0.3, 3.3)),
# ])

# V4 Pip-Net
train_transforms = transforms.Compose([
    transforms.Resize(size=(224+8, 224+8)), 
    TrivialAugmentWideNoColor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224+4, scale=(0.95, 1.)),
    TrivialAugmentWideNoShape(),
    transforms.RandomCrop(size=(224, 224)), #includes crop
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
        

test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406], 
        [0.229, 0.224, 0.225],
    ),
])

In [ ]:
train_dataset = datasets.ImageFolder(
    DATASET_DIR/"train", 
    transform=train_transforms,
)

test_dataset = datasets.ImageFolder(
    DATASET_DIR/"test", 
    transform=test_transforms,
)

In [ ]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
)

# Setup model

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

In [ ]:
model = models.convnext_tiny(
    weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
)
_ = model.to(DEVICE)

In [ ]:
NUM_CLASSES: int = len(train_dataset.classes)
EMBED_DIM: int = 768

In [ ]:
model.classifier.pop(-1)
model.classifier.append(nn.Linear(EMBED_DIM, NUM_CLASSES))
_ = model.to(DEVICE)

# Setup loss & optimizer

In [ ]:
LR: float = 1e-4
WD: float = 1e-4

In [ ]:
criterion = nn.CrossEntropyLoss()


# Train

In [ ]:
from tqdm import tqdm
import pandas as pd

In [ ]:
NUM_PROBE_EPOCHS: int = 12
NUM_FINETUNE_EPOCHS: int = 10
NUM_FULL_EPOCHS: int = 60

LOG_FILEPATH = LOG_DIR/"training_log.csv"
train_log = pd.DataFrame(columns=["epoch", "phase", "train_loss", "train_acc", "test_acc"])

In [ ]:
from torchvision.transforms import v2

cutmix = v2.CutMix(num_classes=NUM_CLASSES)
mixup = v2.MixUp(num_classes=NUM_CLASSES)
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])


@torch.no_grad()
def test_epoch(
    model: nn.Module, 
    data_loader: DataLoader,
):
    model.eval()
    correct, total = 0, 0

    for x, y in tqdm(
        data_loader, 
        total=len(data_loader), 
        desc="Test"
    ):
        x, y = x.to(DEVICE), y.to(DEVICE)
        
        logits = model(x)
        _, y_pred = torch.max(logits, 1)
        correct += (y_pred == y).sum().cpu().item()
        total += y.size(0)

    return correct / total


def train_epoch(
    model: nn.Module, 
    data_loader: DataLoader, 
    criterion: nn.Module, 
    optimizer: torch.optim.Optimizer, 
):
    model.train()

    running_loss = 0.0
    correct, total = 0, 0

    for x, y in tqdm(
        data_loader, 
        total=len(data_loader), 
        desc="Train"
    ):
        x, y = x.to(DEVICE), y.to(DEVICE)
        x, y = cutmix_or_mixup(x, y)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            running_loss += loss.item()
            _, y_pred = torch.max(logits, 1)
            _, y = torch.max(y, 1)
            correct += (y_pred == y).sum().item()
            total += y.size(0)

    train_acc = correct / total
    train_loss = running_loss/len(train_loader)
    return train_loss, train_acc

In [ ]:
def update_optimizer_params(
    phase: str,
    model: nn.Module, 
    optimizer: torch.optim.Optimizer,
):
    trainable_params = []

    if phase == "linear_probe":
        trainable_params = list(model.classifier.parameters())

    elif phase == "finetune":
        trainable_params = list(model.classifier.parameters()) + list(model.features[-1].parameters())

    elif phase == "full":
        trainable_params = list(model.parameters()) 
    else:
        raise NotImplementedError(f"phase {phase} not implemented!")

    for param in model.parameters():
        param.requires_grad = False
    
    for param in trainable_params:
        param.requires_grad = True

    optimizer.param_groups[0]['params'] = trainable_params

In [ ]:
def train_phase(
    phase_name: str,
    num_epochs: int,
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    test_loader: DataLoader,
    train_log: pd.DataFrame,
):
    update_optimizer_params(
        phase=phase_name,
        model=model,
        optimizer=optimizer,
    )
    
    for epoch in range(num_epochs):
        print(f"Phase: {phase_name} | Epoch: {epoch}")
        
        train_loss, train_acc = train_epoch(
            model=model, 
            data_loader=train_loader, 
            criterion=criterion, 
            optimizer=optimizer, 
        )
        print(f"Train: Acc {str(round(train_acc, 3))} | Loss {str(round(train_loss, 3))}")
        
        test_acc = test_epoch(
            model=model,
            data_loader=test_loader,
        )
        print(f"Test: Acc {str(round(test_acc, 3))}")

        # Log results
        curr_log = pd.DataFrame(
            {
                "epoch": [epoch], 
                "phase": [phase_name], 
                "train_loss": [train_loss],
                "train_acc": [train_acc], 
                "test_acc": [test_acc],
            },
        )
    
        train_log = pd.concat([train_log, curr_log])
        train_log.to_csv(LOG_FILEPATH, index=False)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_phase(
    phase_name="linear_probe",
    num_epochs=NUM_PROBE_EPOCHS,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=test_loader,
    train_log=train_log,
)

In [ ]:
# train_phase(
#     phase_name="finetune",
#     num_epochs=NUM_FINETUNE_EPOCHS,
#     model=model,
#     criterion=criterion,
#     optimizer=optimizer,
#     train_loader=train_loader,
#     test_loader=test_loader,
#     train_log=train_log,
# )

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

train_phase(
    phase_name="full",
    num_epochs=NUM_FULL_EPOCHS,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=test_loader,
    train_log=train_log,
)